In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/motion-videos/synced_vids/side/ahmed-hesham-lunges-side.mp4
/kaggle/input/motion-videos/synced_vids/side/abdelrahman-fawaz-sq-side.mp4
/kaggle/input/motion-videos/synced_vids/side/youssef-elghandour-pu-side.mp4
/kaggle/input/motion-videos/synced_vids/side/mohmmed-yasser-lunges-side.mp4
/kaggle/input/motion-videos/synced_vids/side/mohammed-sharshar-lunges-side.mp4
/kaggle/input/motion-videos/synced_vids/side/ahmed-sharshar-pu-side.mp4
/kaggle/input/motion-videos/synced_vids/side/moaz-elsayed-pu-side.mp4
/kaggle/input/motion-videos/synced_vids/side/hossam-elhady-lunges-side.mp4
/kaggle/input/motion-videos/synced_vids/side/ahmed-zakria-pu-side.mp4
/kaggle/input/motion-videos/synced_vids/side/mazen-ehab-sq-side.mp4
/kaggle/input/motion-videos/synced_vids/side/ahmed-wael-shp-side.mp4
/kaggle/input/motion-videos/synced_vids/side/ahmed-hesham-shp-side.mp4
/kaggle/input/motion-videos/synced_vids/side/ahmed-nabil-pu-side.mp4
/kaggle/input/motion-videos/synced_vids/side/ahmed-nabil

In [2]:
!pip install opencv-python mediapipe pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 20.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.20.3
    Uninstalling protobuf-3.20.3:
      Successfully uninstalled protobuf-3.20.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 4.25.8 which is incompatible.
pandas-gbq 0.29.1 requires google-api-core<3.0.0,>=2.10.2, but you have google-api-core 1.34.1 which is incompatible.
google-cloud-storage 2.19.0 requires google-api-core<3.0.0dev,>=2.15.0, but you have google-api-core 1.34

In [3]:
import cv2
import mediapipe as mp
import pandas as pd
import os
from pathlib import Path
import numpy as np

# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=2,
    smooth_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

2025-10-06 05:29:18.046500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759728558.234761      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759728558.290874      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
def extract_keypoints_from_video(video_path):
    """Extract keypoints from a video file"""
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return None

    keypoints_data = []
    frame_number = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Process the frame
        results = pose.process(frame_rgb)

        if results.pose_landmarks:
            # Extract landmarks
            landmarks = results.pose_landmarks.landmark

            # Create a row with all keypoint coordinates
            row_data = {'frame': frame_number}

            for idx, landmark in enumerate(landmarks):
                row_data[f'landmark_{idx}_x'] = landmark.x
                row_data[f'landmark_{idx}_y'] = landmark.y
                row_data[f'landmark_{idx}_z'] = landmark.z
                row_data[f'landmark_{idx}_visibility'] = landmark.visibility

            keypoints_data.append(row_data)

        frame_number += 1

    cap.release()

    if keypoints_data:
        return pd.DataFrame(keypoints_data)
    else:
        return None

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
def get_motion_and_view_from_filename(filename):
    """
    Extract motion type and view from filename
    Expected format: contains keywords like 'squad', 'lungs', 'pushup', 'shp'
    and 'front' or 'side'
    """
    filename_lower = filename.lower()

    # Detect motion
    motion = None
    if 'sq' in filename_lower or 'squat' in filename_lower:
        motion = 'sq'
    elif 'lunges' in filename_lower:
        motion = 'lunges'
    elif 'pu' in filename_lower or 'push_up' in filename_lower or 'push-up' in filename_lower:
        motion = 'pu'
    elif 'shp' in filename_lower or 'shoulder' in filename_lower:
        motion = 'shp'

    # Detect view
    view = None
    if 'front' in filename_lower:
        view = 'front'
    elif 'side' in filename_lower:
        view = 'side'

    return motion, view

def process_videos_from_drive(drive_path, output_base_dir):

    # Create output directory structure
    views = ['front', 'side']
    motions = ['sq', 'lunges', 'pu', 'shp']

    for view in views:
        for motion in motions:
            output_dir = os.path.join(output_base_dir, view, motion)
            os.makedirs(output_dir, exist_ok=True)

    # Supported video extensions
    video_extensions = ['.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv']

    # Get all video files from drive path
    video_files = []
    for ext in video_extensions:
        video_files.extend(Path(drive_path).rglob(f'*{ext}'))

    print(f"Found {len(video_files)} video files")

    # Process each video
    for video_path in video_files:
        filename = video_path.name
        print(f"\nProcessing: {filename}")

        # Get motion and view from filename
        motion, view = get_motion_and_view_from_filename(filename)

        if motion is None:
            print(f"  ⚠ Could not detect motion type in filename. Skipping.")
            continue

        if view is None:
            print(f"  ⚠ Could not detect view type in filename. Skipping.")
            continue

        print(f"  Detected: Motion={motion}, View={view}")

        # Extract keypoints
        df_keypoints = extract_keypoints_from_video(str(video_path))

        if df_keypoints is not None and len(df_keypoints) > 0:
            # Create output path
            output_dir = os.path.join(output_base_dir, view, motion)
            output_filename = f"{Path(filename).stem}_keypoints.csv"
            output_path = os.path.join(output_dir, output_filename)

            # Save to CSV
            df_keypoints.to_csv(output_path, index=False)
            print(f"  ✓ Saved {len(df_keypoints)} frames to: {output_path}")
        else:
            print(f"  ✗ No keypoints extracted from this video")

    print("\n" + "="*60)
    print("Processing complete!")
    print("="*60)

In [6]:
if __name__ == "__main__":
    # Set your paths here
    DRIVE_PATH = "/kaggle/input/motion-videos/synced_vids"  # Change this to your video folder
    OUTPUT_DIR = "PhysicsMotion-Results"  # Change this to your desired output folder

    # Process all videos
    process_videos_from_drive(DRIVE_PATH, OUTPUT_DIR)

    # Close MediaPipe
    pose.close()

    print("\n📊 Summary of extracted data:")
    print(f"Output directory: {OUTPUT_DIR}")
    print("\nDirectory structure:")
    for view in ['front', 'side']:
        for motion in ['Squad', 'Lungs', 'PushUp', 'SHP']:
            path = os.path.join(OUTPUT_DIR, view, motion)
            if os.path.exists(path):
                num_files = len([f for f in os.listdir(path) if f.endswith('.csv')])
                print(f"  {view}/{motion}: {num_files} CSV files")

W0000 00:00:1759728571.430042      71 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1759728571.606428      70 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Found 390 video files

Processing: ahmed-hesham-lunges-side.mp4
  Detected: Motion=lunges, View=side


W0000 00:00:1759728572.040383      70 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


  ✓ Saved 721 frames to: PhysicsMotion-Results/side/lunges/ahmed-hesham-lunges-side_keypoints.csv

Processing: abdelrahman-fawaz-sq-side.mp4
  Detected: Motion=sq, View=side
  ✓ Saved 240 frames to: PhysicsMotion-Results/side/sq/abdelrahman-fawaz-sq-side_keypoints.csv

Processing: youssef-elghandour-pu-side.mp4
  Detected: Motion=pu, View=side
  ✓ Saved 335 frames to: PhysicsMotion-Results/side/pu/youssef-elghandour-pu-side_keypoints.csv

Processing: mohmmed-yasser-lunges-side.mp4
  Detected: Motion=lunges, View=side
  ✓ Saved 690 frames to: PhysicsMotion-Results/side/lunges/mohmmed-yasser-lunges-side_keypoints.csv

Processing: mohammed-sharshar-lunges-side.mp4
  Detected: Motion=lunges, View=side
  ✓ Saved 601 frames to: PhysicsMotion-Results/side/lunges/mohammed-sharshar-lunges-side_keypoints.csv

Processing: ahmed-sharshar-pu-side.mp4
  Detected: Motion=pu, View=side
  ✓ Saved 420 frames to: PhysicsMotion-Results/side/pu/ahmed-sharshar-pu-side_keypoints.csv

Processing: moaz-elsayed